In [2]:
import re
import pandas as pd
import geopandas as gpd
from shapely import wkt
from pathlib import Path

# Paths
csv_path = Path("../../data/venues/tac-list/venues_TAC_EDDIT V2 March 12.csv")
gpkg_path = csv_path.parent / "venues_TAC.gpkg"

# Load CSV
df = pd.read_csv(csv_path, dtype=str)

def parse_wkt(v):
    """Parse a WKT string, fixing missing commas between coordinate pairs."""
    if not pd.notna(v) or not str(v).strip():
        return None
    v = str(v).strip()
    # Some WKTs lack commas between coordinate pairs (e.g. "lon lat lon lat").
    # In this dataset lons are negative (~-79) and lats positive (~43),
    # so insert a comma wherever a positive decimal is followed by a negative one.
    v = re.sub(r'(\d) (-\d)', r'\1, \2', v)
    try:
        return wkt.loads(v)
    except Exception:
        return None

# Parse WKT geometry
df["geometry"] = df["geometry_wkt"].apply(parse_wkt)

# Drop the original WKT column
df = df.drop(columns=["geometry_wkt"])

# Convert to GeoDataFrame (CRS = WGS 84)
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

# Save to GeoPackage
gdf.to_file(gpkg_path, driver="GPKG")

print(f"Saved {len(gdf)} features to {gpkg_path.resolve()}")
gdf.head()


Saved 50 features to /home/aniket/Programming/eddit-tac/data/venues/tac-list/venues_TAC.gpkg


,internal_venue_id,venue_name,address,postal_code,osm_type,osm_id,lat,lon,polygon_osm_id,polygon_type,...,building_dedication,match_method,community_arts,dance,literary,music,theatre,visual_media_arts,arts_programming,geometry
0,480402,Chinese Cultural Centre of Greater Toronto,5183 Sheppard Avenue East,M1B 5Z5,way,660356849,43.7944585,-79.2340598,660356849,yes,...,dedicated,name+address,low,mid,rare,high,rare,none,very_high,"POLYGON ((-79.23420 43.79426, -79.23422 43.794..."
1,545884,Daniels Spectrum,585 Dundas Street East,M5A 2B7,node,2898358868,43.6601096,-79.3619394,229563439,yes,...,dedicated,name+address,very_high,very_high,mid,very_high,very_high,very_high,very_high,"POLYGON ((-79.36280 43.66002, -79.36220 43.660..."
2,480436,Four Seasons Centre for the Performing Arts,145 Queen Street West,M5H 4G1,way,6694663,43.6505983,-79.385527,6694663,yes,...,dedicated,name+address,none,high,none,very_high,rare,none,very_high,"POLYGON ((-79.38599 43.65026, -79.38584 43.650..."
3,480480,Massey Hall,178 Victoria Street,M5B 1T6,way,62004062,43.6539666,-79.3789815,62004062,yes,...,dedicated,name+address,low,rare,none,very_high,rare,rare,very_high,"POLYGON ((-79.37885 43.65422, -79.37868 43.653..."
4,638721,Museum of Contemporary Art Toronto Canada (MOCA),158 Sterling Road,M6R 2B2,way,490218078,43.6546142,-79.4452026,490218078,yes,...,dedicated,address_only,rare,rare,rare,rare,rare,mid,high,"POLYGON ((-79.44540 43.65480, -79.44516 43.654..."
